# HADDOCK3 Install and Example

## Here are possible install routes

### Normal way

In [ ]:
!conda create -y -n haddock python=3.13

Activate haddocak

In [ ]:
!conda install -y -c conda-forge c-compiler cxx-compiler
%pip install --upgrade pip setuptools
#%pip install freesasa
%pip install haddock3

Or with MPI4py

In [ ]:
%pip install 'haddock3[mpi]'

### From the source

In [ ]:
!git clone https://github.com/haddocking/haddock3.git
!cd haddock3
!pip install .

## Running HADDOCK

## HADDOCK3 Workflow
This cell handles PDB preprocessing (chain naming) and subdirectory archiving automatically.

In [7]:
import os
import shutil
from datetime import datetime
from pathlib import Path
from haddock.clis.cli import main

# === CONFIGURATION ===
mol1_path = "../Data/TIMP3_Xray.pdb"
mol2_path = "../Data/MMP2_Xray.pdb"
files_dir = "../Local/haddock_data"
run_dir = "haddock3_test_run"
past_runs_base = f"{files_dir}/past_runs"
# =====================

def modify_chain(in_file, out_file, new_chain):
    with open(in_file, 'r') as f_in, open(out_file, 'w') as f_out:
        for line in f_in:
            if line.startswith(('ATOM  ', 'HETATM', 'TER   ')):
                if len(line) >= 22:
                    f_out.write(line[:21] + new_chain + line[22:])
                else:
                    f_out.write(line)
            else:
                f_out.write(line)

if os.path.exists(run_dir):
    raise("HADDOCK Output folder must be empty!")

# 2. Prepare structures with distinct chains
os.makedirs(files_dir, exist_ok=True)
p1 = os.path.join(files_dir, "mol1_fixed.pdb")
p2 = os.path.join(files_dir, "mol2_fixed.pdb")
modify_chain(mol1_path, p1, "A")
modify_chain(mol2_path, p2, "B")

# 3. Create HADDOCK3 config
config_content = f"""
run_dir = \"{run_dir}\"

molecules = [
    \"{p1}\",
    \"{p2}\"
]

[topoaa]
[rigidbody]
sampling = 10
cmrest = true
[caprieval]
"""
with open(f"{files_dir}/run.cfg", "w") as f:
    f.write(config_content)

print("Starting HADDOCK3 run...")
# 4. Run HADDOCK3
main(f"{files_dir}/run.cfg")

# Move output
now = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
archive_path = os.path.join(past_runs_base, now)
os.makedirs(archive_path, exist_ok=True)
print(f"Moving existing run to {archive_path}")
shutil.move(run_dir, archive_path)

Starting HADDOCK3 run...
[2026-02-27 09:27:30,969 cli INFO] 
##############################################
#                                            #
#                 HADDOCK3                   #
#                                            #
##############################################

!! Some of the HADDOCK3 components use CNS (Crystallographic and NMR System) which is free of use for non-profit applications. !!
!! For commercial use it is your own responsibility to have a proper license. !!
!! For details refer to the DISCLAIMER file in the HADDOCK3 repository. !!

Starting HADDOCK3 v2025.11.0 on 2026-02-27 09:27:00

Python 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]

[2026-02-27 09:27:31,345 libworkflow INFO] Reading instructions step 0_topoaa
[2026-02-27 09:27:31,348 libworkflow INFO] Reading instructions step 1_rigidbody
[2026-02-27 09:27:31,349 libworkflow INFO] Reading instructions step 2_caprieval
[2026-02-27 09:27:31,376 base_cns_module INFO] Running [topoaa] 

'../Local/haddock_data/past_runs/2026-02-27_09-28-04/haddock3_test_run'